# 02 — MCP in one notebook process

**Learning goal:** define MCP tools, discover and call them through the real MCP protocol over an in-memory transport, then optionally let a local model choose tools.

**Prerequisites:** Python 3 and this notebook inside `demos/`. Run cells top to bottom. `PROFILE` remains a shared suite switch (`"onia"` or `"devtalks"`) although this protocol-only notebook uses the same tools for both. The direct MCP cells need neither LM Studio nor internet and run by default. Only the final agent cell requires LM Studio; it is guarded by `RUN_LM_STUDIO_DEMO=False`. Re-running definitions is safe because `make_mcp_server()` creates a fresh server each time.

In [ ]:
%pip install -q openai==2.53.0 "mcp[cli]==2.0.0"

## 1. Shared configuration
Constants are editable and environment variables override them. The document-root check catches a notebook launched from an unrelated working directory even though this example does not read the corpus.

In [ ]:
import os
from pathlib import Path

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "lm-studio")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen/qwen3.5-9b")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-qwen3-embedding-4b")
PROFILE = os.getenv("PROFILE", "devtalks").lower()
TOP_K = int(os.getenv("TOP_K", "3"))
RUN_DIRECT_MCP_DEMO = True
RUN_LM_STUDIO_DEMO = False

if PROFILE not in {"onia", "devtalks"}:
    raise ValueError("PROFILE must be 'onia' or 'devtalks'.")
if TOP_K <= 0:
    raise ValueError("TOP_K must be a positive integer.")

def find_demo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd / "demos", *cwd.parents]:
        if (candidate / "documents" / "shared" / "mcp_overview.md").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find demos/documents. Launch Jupyter from the presentations root or demos directory."
    )

DEMO_ROOT = find_demo_root()
print(f"Profile: {PROFILE} | demo root: {DEMO_ROOT}")

## 2. Define a fresh MCP server and two tools
The factory prevents duplicate tool registration when this cell is re-run. `current_time` accepts standard IANA timezone names.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from mcp.server import MCPServer

def make_mcp_server() -> MCPServer:
    server = MCPServer("notebook-mcp-demo")

    @server.tool()
    def add(a: int, b: int) -> int:
        """Return the sum of two integers."""
        return a + b

    @server.tool()
    def current_time(timezone: str = "UTC") -> str:
        """Return the current wall-clock time in an IANA timezone."""
        try:
            tz = ZoneInfo(timezone)
        except ZoneInfoNotFoundError:
            return f"Unknown timezone: {timezone!r}"
        return datetime.now(tz).isoformat(timespec="seconds")

    return server

mcp_server = make_mcp_server()
print("Fresh MCPServer created.")

## 3. Discover and call tools through MCP — **offline**
This is not a direct Python function call: `ClientSession` initializes an MCP connection, asks the server for its schemas, and sends tool-call requests through `InMemoryTransport`. Jupyter supports the top-level `await` used here.

In [ ]:
from typing import Any

from mcp import ClientSession
from mcp.client._memory import InMemoryTransport

def tool_result_text(result: Any) -> str:
    return "\n".join(
        getattr(block, "text", str(block)) for block in result.content
    ).strip()

if RUN_DIRECT_MCP_DEMO:
    async with InMemoryTransport(mcp_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            advertised = await session.list_tools()
            print("Advertised tools:")
            for tool in advertised.tools:
                print(f"  {tool.name}: {tool.description}")
            sum_result = await session.call_tool("add", {"a": 17, "b": 25})
            time_result = await session.call_tool(
                "current_time", {"timezone": "Europe/Bucharest"}
            )
            print("add =>", tool_result_text(sum_result))
            print("current_time =>", tool_result_text(time_result))
else:
    print("Direct MCP demonstration skipped.")

## 4. Optional model-driven tool loop — **requires LM Studio**
The client translates MCP tool descriptors into OpenAI-compatible function schemas. The model chooses a tool; the client executes it through MCP and returns the result for a final answer. No internet is needed.

In [ ]:
import json
from openai import OpenAI

def openai_tool_schemas(mcp_tools: Any) -> list[dict]:
    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema or {"type": "object"},
            },
        }
        for tool in mcp_tools.tools
    ]

QUESTION = "What is 17 plus 25, and what time is it in Europe/Bucharest?"

if RUN_LM_STUDIO_DEMO:
    llm = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    agent_server = make_mcp_server()
    async with InMemoryTransport(agent_server, raise_exceptions=True) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            schemas = openai_tool_schemas(await session.list_tools())
            messages = [
                {"role": "system", "content": "Use MCP tools when helpful; answer concisely."},
                {"role": "user", "content": QUESTION},
            ]
            for step in range(5):
                response = llm.chat.completions.create(
                    model=CHAT_MODEL, messages=messages, tools=schemas,
                    tool_choice="auto", temperature=0.2,
                )
                message = response.choices[0].message
                calls = message.tool_calls or []
                if not calls:
                    print(message.content or "")
                    break
                messages.append({
                    "role": "assistant",
                    "content": message.content or "",
                    "tool_calls": [call.model_dump() for call in calls],
                })
                for call in calls:
                    arguments = json.loads(call.function.arguments or "{}")
                    result = await session.call_tool(call.function.name, arguments)
                    text = tool_result_text(result)
                    print(f"tool: {call.function.name}({arguments}) -> {text}")
                    messages.append({"role": "tool", "tool_call_id": call.id, "content": text})
            else:
                print("Stopped after five tool-selection steps.")
else:
    print("Skipped model-driven agent. Set RUN_LM_STUDIO_DEMO=True to enable it.")